# Day 4: Pydantic Structured Outputs & Automated Tool Dispatch
### Learning Objectives & Success Criteria
- Constrain SLM output into strict machine-readable JSON validated by Pydantic (`ActionTicket`).
- Execute deterministic downstream tool calls using externalized webhook endpoints (`SETTINGS.ops_webhook_base_url`).
- Trace `RETRIEVER` → `LLM` → `TOOL` spans in Arize Phoenix.


### Cell 1 — Centralized Settings & Modular Engine Bootstrap


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from engine.settings import SETTINGS
from engine.router import ModelRouter
from engine.qdrant_store import QdrantVectorStore
from engine.use_case_loader import UseCaseLoader
from engine.visualizer import StoryVisualizer
from engine.debugger import TraceDebugger
from engine.quiz_helper import render_quiz

# Import PromptRegistry safely regardless of module filename (prompts.py or prompt_registry.py)
try:
    from engine.prompts import PromptRegistry
except ImportError:
    from engine.prompt_registry import PromptRegistry

# All URLs, paths, domain names, and cloud toggles come from SETTINGS (.env + configs/app_config.yaml)
# To test your own domain, see CUSTOM_USE_CASE_GUIDE.md and set ACTIVE_DOMAIN = "your_domain"
ACTIVE_DOMAIN = SETTINGS.active_domain

loader = UseCaseLoader(domain=ACTIVE_DOMAIN)
llm = ModelRouter(local_model_path=SETTINGS.local_model_path, enable_vertex=SETTINGS.enable_vertex_ai)
qdrant = QdrantVectorStore(collection_name=loader.collection_name, path=SETTINGS.qdrant_db_path)
prompts = PromptRegistry()
viz = StoryVisualizer()
dbg = TraceDebugger(debug_mode=False, enable_phoenix=False)

RUNBOOKS = loader.load_runbooks()
INCIDENTS = loader.load_incidents()
qdrant.upsert_documents(RUNBOOKS)

print(f"[Ready] Domain: {loader.domain} | LLM: {llm.provider} | VectorDB: {qdrant.mode} ({loader.collection_name})")


### Cell 2 — Production Incident Selector (0..9)


In [ ]:
# Select any of the 10 production incidents in the active domain (0..9)
INCIDENT_INDEX = 0
ACTIVE_INCIDENT = INCIDENTS[INCIDENT_INDEX]

print(f"Selected Incident: [{ACTIVE_INCIDENT['incident_id']}] {ACTIVE_INCIDENT['title']} (Severity: {ACTIVE_INCIDENT['severity']})")
print("Raw Log:", ACTIVE_INCIDENT["raw_log"])


### Cell 3 — Engineering Concept Brief
**Concept: Schema-Constrained JSON & Externalized Webhook Dispatch**
Enforces Pydantic `ActionTicket` JSON output from the local SLM and builds the remediation webhook URL from `SETTINGS.ops_webhook_base_url`.


### Cell 4 — Traced Execution & X-Ray Visual Storyboard


In [ ]:
from pydantic import BaseModel, Field

class ActionTicket(BaseModel):
    incident_id: str = Field(description="Incident ID")
    severity: str = Field(description="P1, P2, or P3")
    service: str = Field(description="Impacted service name")
    runbook_id: str = Field(description="Matched runbook ID")

def dispatch_remediation(ticket: ActionTicket) -> dict:
    webhook_url = f"{SETTINGS.ops_webhook_base_url.rstrip('/')}/{ticket.runbook_id}"
    return {"status": "DISPATCHED", "webhook": webhook_url, "target_service": ticket.service, "priority": ticket.severity}

dbg.start_run("day04_structured")
with dbg.start_span(f"Day4_StructuredTools_{ACTIVE_INCIDENT['incident_id']}", span_kind="CHAIN") as root_span:
    t0 = time.perf_counter()
    top_rb = qdrant.search(ACTIVE_INCIDENT["raw_log"], top_k=1)[0]
    t_ret = round((time.perf_counter() - t0) * 1000, 2)
    dbg.record_step("qdrant_context_lookup", ACTIVE_INCIDENT["raw_log"], top_rb, t_ret, span_kind="RETRIEVER")

    p_cfg = prompts.load("day04_json_extractor")
    u_prompt = prompts.render(p_cfg["user_template"], incident_text=ACTIVE_INCIDENT["raw_log"], runbook_context=top_rb["text"])
    gen = llm.generate(prompt=u_prompt, system_instruction=p_cfg["system_prompt"], temperature=0.0, max_tokens=180, response_schema=ActionTicket)
    parsed_ticket = ActionTicket.model_validate_json(gen["output"])
    dbg.record_step("qwen_json_extraction", u_prompt, parsed_ticket.model_dump(), gen["latency_ms"], span_kind="LLM")

    t2 = time.perf_counter()
    dispatch_res = dispatch_remediation(parsed_ticket)
    t_tool = round((time.perf_counter() - t2) * 1000, 2)
    dbg.record_step("dispatch_remediation_tool", parsed_ticket.model_dump(), dispatch_res, t_tool, span_kind="TOOL")

viz.render_story(
    title=f"Day 4 Structured JSON & Tool Execution — {ACTIVE_INCIDENT['incident_id']}",
    stages=[
        {"name": "Qdrant Top-1 Runbook", "badge": "RETRIEVER", "latency_ms": t_ret, "summary": top_rb["metadata"]["runbook_id"], "payload": top_rb["text"]},
        {"name": "Pydantic Validated Ticket", "badge": "LLM SCHEMA", "latency_ms": gen["latency_ms"], "summary": "Strict JSON contract", "payload": parsed_ticket.model_dump()},
        {"name": "Externalized Webhook Dispatch", "badge": "TOOL CALL", "latency_ms": t_tool, "summary": dispatch_res["webhook"], "payload": dispatch_res}
    ],
    phoenix_url=dbg.phoenix_url
)


### Cell 5 — Guarded Cloud Parity Check (`SETTINGS.enable_vertex_ai`)


In [ ]:
if SETTINGS.enable_vertex_ai:
    print("Vertex AI native JSON response_schema active.")
else:
    print("ENABLE_VERTEX_AI=false -> Local Pydantic schema + externalized webhook verified.")


### Cell 6 — Automated Validation Gate


In [ ]:
assert parsed_ticket.severity in ("P1", "P2", "P3") and dispatch_res["status"] == "DISPATCHED"
print("[PASS] Day 4 Structured Outputs Gate passed!")


### Cell 7 — Active Lab Experiment
**Hands-On Lab Challenge:** Add an `auto_approved: bool = Field(description='True only for P3')` field to `ActionTicket` and re-run Cell 4 to see `ModelRouter` dynamically inject the new JSON schema key into Qwen's system prompt.


### Cell 8 — Interactive Self-Assessment Questionnaire


In [ ]:
import json, os, sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from engine.quiz_helper import render_quiz

with open(repo_root / "configs" / "quizzes" / "day04.json", "r", encoding="utf-8") as f:
    quiz_cfg = json.load(f)

render_quiz(quiz_cfg["title"], quiz_cfg["questions"])


### Cell 9 — Bring Your Own Use Case (BYOUC)
Want to run all 10 notebooks against your own enterprise domain? Follow **[CUSTOM_USE_CASE_GUIDE.md](../CUSTOM_USE_CASE_GUIDE.md)** (`python scripts/scaffold_domain.py --domain your_domain --prefix YOUR-`) and set `ACTIVE_DOMAIN = "your_domain"` in Cell 1.
